<a href="https://colab.research.google.com/github/sara-iqbal/Bond-Index-Replication-Rebalancing-Engine/blob/main/bond_index_rebalancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 Bond Index Replication & Rebalancing Engine
### BlackRock-Style Fixed Income ETF Portfolio Construction Algorithm

**Author:** Sara Iqbal | MSc Data Science | Python Engineer Portfolio Project

---

## Architecture
```
Benchmark Index (Real iShares weights)
        ↓
Synthetic Bond Universe (500 bonds, realistic parameters)
        ↓
Portfolio Construction Optimiser (SciPy SLSQP)
        ↓
Constraint Engine (Duration, Turnover, Liquidity, Position Limits)
        ↓
Monthly Rebalancing Event → Auto Order List Generation
        ↓
Analytics: Tracking Error, DV01, Yield, Credit Quality, Sharpe
```

**Key Metrics Reported:**
- Annualised Tracking Error vs benchmark
- Portfolio DV01 (Dollar Value of 1bp move)
- Modified Duration mismatch
- Yield-to-Maturity vs benchmark
- Credit quality distribution
- Monthly turnover %
- Rebalancing order lists (buy/sell/hold)
- Sharpe Ratio of replication strategy

In [1]:
#  Step 1: Install Dependencies
!pip install scipy numpy pandas plotly openpyxl -q

In [2]:
#  Step 2: Import Libraries
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('✅ Libraries loaded')

✅ Libraries loaded


In [3]:
#  Step 3: Real iShares Benchmark Weights (iBoxx USD Liquid Investment Grade)
# Sector weights sourced from iShares iBoxx $ Investment Grade Corporate Bond ETF (LQD)
# Published holdings data — BlackRock website (public)

BENCHMARK_SECTORS = {
    'Financials':        0.28,   # Banks, Insurance, REITs
    'Industrials':       0.22,   # Manufacturing, Energy, Materials
    'Utilities':         0.08,   # Electric, Gas, Water
    'Consumer':          0.14,   # Consumer Discretionary + Staples
    'Technology':        0.12,   # Tech, Telecom
    'Healthcare':        0.09,   # Pharma, Biotech, Medical
    'Government':        0.07,   # Agencies, Supranationals
}

BENCHMARK_RATINGS = {
    'AAA': 0.05,
    'AA':  0.12,
    'A':   0.38,
    'BBB': 0.45,
}

BENCHMARK_MATURITY_BUCKETS = {
    '1-3Y':   0.18,
    '3-5Y':   0.22,
    '5-7Y':   0.20,
    '7-10Y':  0.24,
    '10Y+':   0.16,
}

# Benchmark-level analytics (LQD as of 2024)
BENCHMARK_DURATION   = 8.42   # Modified Duration (years)
BENCHMARK_YTM        = 5.31   # Yield to Maturity (%)
BENCHMARK_DV01       = 0.0842  # Per $1000 face value
BENCHMARK_N_BONDS    = 2800   # ~2,800 constituents in LQD

print('✅ Benchmark weights loaded (iShares LQD proxy)')
print(f'   Modified Duration: {BENCHMARK_DURATION}y | YTM: {BENCHMARK_YTM}% | Constituents: {BENCHMARK_N_BONDS}')

✅ Benchmark weights loaded (iShares LQD proxy)
   Modified Duration: 8.42y | YTM: 5.31% | Constituents: 2800


In [4]:
#  Step 4: Synthetic Bond Universe Generation
# 500 synthetic bonds with realistic fixed income parameters
# Distributions calibrated to actual investment-grade corporate bond markets

N_BONDS = 500
SECTORS = list(BENCHMARK_SECTORS.keys())
RATINGS = ['AAA', 'AA', 'A', 'BBB']
CURRENCIES = ['USD', 'EUR', 'GBP', 'JPY']
COUNTRIES  = ['US', 'UK', 'DE', 'FR', 'JP', 'CA', 'AU', 'NL', 'CH', 'SE']

def rating_to_spread(rating):
    """OAS spread by credit rating (basis points, calibrated to 2024 IG market)"""
    spreads = {'AAA': (20, 15), 'AA': (45, 20), 'A': (90, 35), 'BBB': (145, 55)}
    mu, sigma = spreads[rating]
    return max(5, np.random.normal(mu, sigma))

def rating_to_default_prob(rating):
    """Annual probability of default by rating (Moody's historical)"""
    probs = {'AAA': 0.0001, 'AA': 0.0003, 'A': 0.0010, 'BBB': 0.0024}
    return probs[rating]

# Assign ratings using benchmark distribution
rating_weights = list(BENCHMARK_RATINGS.values())
bond_ratings = np.random.choice(RATINGS, size=N_BONDS, p=rating_weights)

# Assign sectors using benchmark distribution
sector_weights = list(BENCHMARK_SECTORS.values())
bond_sectors = np.random.choice(SECTORS, size=N_BONDS, p=sector_weights)

# Generate maturity dates (1–30 years)
maturities = np.random.choice(
    [1.5, 2.5, 4.0, 6.0, 8.5, 10.0, 15.0, 20.0, 30.0],
    size=N_BONDS,
    p=[0.08, 0.12, 0.18, 0.18, 0.14, 0.13, 0.08, 0.05, 0.04]
)

# Coupons: risk-free rate + spread
risk_free = 4.35  # US 10Y Treasury yield (2024)
oas_spreads = np.array([rating_to_spread(r) for r in bond_ratings]) / 100
coupons = risk_free + oas_spreads * 100 + np.random.normal(0, 0.3, N_BONDS)
coupons = np.clip(coupons, 1.5, 9.5)

# Modified Duration (approximation: duration ≈ maturity for zero-coupon, less for coupon bonds)
durations = maturities * (1 - 0.1 * coupons / 10) + np.random.normal(0, 0.2, N_BONDS)
durations = np.clip(durations, 0.5, 25)

# YTM = coupon + spread
ytms = coupons + np.random.normal(0, 0.15, N_BONDS)
ytms = np.clip(ytms, 1.0, 10.0)

# Price (par = 100, moves inversely with yield)
prices = 100 + np.random.normal(0, 5, N_BONDS) - (ytms - 5.0) * durations
prices = np.clip(prices, 70, 130)

# Liquidity score (0–1, higher = more liquid)
# AAA/AA bonds and shorter maturities are more liquid
rating_liq = {'AAA': 0.9, 'AA': 0.8, 'A': 0.65, 'BBB': 0.5}
liquidity = np.array([rating_liq[r] for r in bond_ratings]) + np.random.uniform(-0.15, 0.15, N_BONDS)
liquidity = np.clip(liquidity, 0.1, 1.0)

# DV01 per $1000 face (dollar value of 1 basis point)
dv01 = (durations * prices * 0.0001)  # DV01 = Duration × Price × 0.0001

# Face value outstanding ($M) — proxy for index eligibility
face_value = np.random.lognormal(mean=5.5, sigma=1.0, size=N_BONDS)  # ~$245M average
face_value = np.clip(face_value, 50, 5000)

# Build DataFrame
bonds = pd.DataFrame({
    'bond_id':      [f'BOND_{i:04d}' for i in range(N_BONDS)],
    'issuer':       [f'{s[:3].upper()}{i:03d} Corp' for i, s in enumerate(bond_sectors)],
    'sector':       bond_sectors,
    'rating':       bond_ratings,
    'country':      np.random.choice(COUNTRIES, N_BONDS, p=[0.45,0.12,0.10,0.09,0.07,0.06,0.04,0.03,0.02,0.02]),
    'currency':     np.random.choice(CURRENCIES, N_BONDS, p=[0.72,0.14,0.10,0.04]),
    'maturity_yrs': np.round(maturities, 1),
    'coupon_pct':   np.round(coupons, 2),
    'ytm_pct':      np.round(ytms, 2),
    'price':        np.round(prices, 2),
    'mod_duration': np.round(durations, 2),
    'dv01':         np.round(dv01, 4),
    'oas_bps':      np.round(oas_spreads * 100, 1),
    'liquidity':    np.round(liquidity, 3),
    'face_value_m': np.round(face_value, 1),
    'default_prob': [rating_to_default_prob(r) for r in bond_ratings],
})

print(f'✅ Bond universe: {len(bonds)} synthetic bonds generated')
print(f'   Duration range: {bonds.mod_duration.min():.1f}y – {bonds.mod_duration.max():.1f}y')
print(f'   YTM range: {bonds.ytm_pct.min():.2f}% – {bonds.ytm_pct.max():.2f}%')
print(f'   Rating distribution:')
print(bonds.rating.value_counts().to_string())
bonds.head()

✅ Bond universe: 500 synthetic bonds generated
   Duration range: 0.9y – 25.0y
   YTM range: 8.79% – 10.00%
   Rating distribution:
rating
BBB    228
A      175
AA      66
AAA     31


,bond_id,issuer,sector,rating,country,currency,maturity_yrs,coupon_pct,ytm_pct,price,mod_duration,dv01,oas_bps,liquidity,face_value_m,default_prob
0,BOND_0000,CON000 Corp,Consumer,A,US,JPY,2.5,9.5,9.32,88.15,2.04,0.0180,5.0,0.536,493.1,0.0010
1,BOND_0001,UTI001 Corp,Utilities,BBB,US,USD,6.0,9.5,9.37,78.36,5.48,0.0429,271.0,0.444,4083.3,0.0024
2,BOND_0002,IND002 Corp,Industrials,BBB,US,USD,15.0,9.5,9.59,70.00,13.67,0.0957,68.6,0.591,220.4,0.0024
3,BOND_0003,TEC003 Corp,Technology,BBB,AU,USD,10.0,9.5,9.36,70.00,9.28,0.0649,54.5,0.608,362.6,0.0024
4,BOND_0004,CON004 Corp,Consumer,AA,DE,USD,10.0,9.5,9.57,70.00,9.37,0.0656,65.5,0.691,117.0,0.0003


In [5]:
#  Step 5: Portfolio Construction Optimiser
# Minimise tracking error vs benchmark subject to:
#   1. Fully invested (weights sum to 1)
#   2. No short selling (weights >= 0)
#   3. Max single position: 2%
#   4. Min liquidity score: 0.3
#   5. Duration within ±0.5y of benchmark
#   6. Max sector deviation: ±5% from benchmark
#   7. Max turnover per rebalance: 15%

PORTFOLIO_SIZE   = 150    # Replicate 2800-bond index with 150 bonds (stratified sampling)
MAX_WEIGHT       = 0.02   # 2% max single position
MIN_LIQUIDITY    = 0.30   # Minimum liquidity score
DURATION_BAND    = 0.50   # ±0.5 years vs benchmark duration
MAX_SECTOR_DEV   = 0.05   # ±5% sector deviation from benchmark
MAX_TURNOVER     = 0.15   # 15% max monthly turnover
PORTFOLIO_NAV    = 100_000_000  # $100M portfolio

def select_eligible_bonds(universe, min_liq=MIN_LIQUIDITY, min_face=100):
    """Filter to investable universe"""
    eligible = universe[
        (universe.liquidity >= min_liq) &
        (universe.face_value_m >= min_face)
    ].copy()
    return eligible

def stratified_sample(eligible, n=PORTFOLIO_SIZE):
    """Stratified sampling to ensure sector/rating/maturity coverage"""
    sampled = []
    # Sample proportionally from each sector
    for sector, weight in BENCHMARK_SECTORS.items():
        sector_bonds = eligible[eligible.sector == sector]
        n_sector = max(1, int(n * weight))
        if len(sector_bonds) >= n_sector:
            # Prefer higher liquidity bonds
            sector_sample = sector_bonds.nlargest(n_sector * 3, 'liquidity').sample(
                min(n_sector, len(sector_bonds)), random_state=42
            )
            sampled.append(sector_sample)
    selected = pd.concat(sampled).drop_duplicates('bond_id').head(n)
    return selected.reset_index(drop=True)

def compute_tracking_error(weights, portfolio_bonds, benchmark_stats):
    """Tracking error = std dev of daily return differences vs benchmark"""
    port_duration = np.dot(weights, portfolio_bonds.mod_duration)
    port_ytm      = np.dot(weights, portfolio_bonds.ytm_pct)

    # Duration mismatch contributes to TE via interest rate risk
    duration_mismatch = port_duration - benchmark_stats['duration']
    ytm_mismatch      = port_ytm - benchmark_stats['ytm']

    # Spread dispersion within portfolio
    spread_dispersion = np.std(portfolio_bonds.oas_bps * weights)

    # Tracking error approximation (annualised bps)
    te = np.sqrt(
        (duration_mismatch * 100) ** 2 +  # Duration mismatch in bps
        (ytm_mismatch * 50) ** 2 +         # Yield mismatch
        spread_dispersion ** 2              # Spread dispersion
    )
    return te

def optimise_portfolio(portfolio_bonds, prev_weights=None):
    """SciPy SLSQP optimiser — minimise tracking error subject to constraints"""
    n = len(portfolio_bonds)
    benchmark_stats = {
        'duration': BENCHMARK_DURATION,
        'ytm':      BENCHMARK_YTM,
        'dv01':     BENCHMARK_DV01,
    }

    # Initial weights: equal weight as starting point
    w0 = np.ones(n) / n if prev_weights is None else prev_weights

    constraints = [
        # Fully invested
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0},
        # Duration constraint: within ±0.5y of benchmark
        {'type': 'ineq', 'fun': lambda w: DURATION_BAND - abs(
            np.dot(w, portfolio_bonds.mod_duration) - BENCHMARK_DURATION)},
    ]

    # Sector constraints: ±5% from benchmark
    for sector, bm_weight in BENCHMARK_SECTORS.items():
        sector_mask = (portfolio_bonds.sector == sector).values.astype(float)
        constraints.append({
            'type': 'ineq',
            'fun': lambda w, m=sector_mask, bw=bm_weight: MAX_SECTOR_DEV - abs(np.dot(w, m) - bw)
        })

    # Turnover constraint (if rebalancing)
    if prev_weights is not None:
        constraints.append({
            'type': 'ineq',
            'fun': lambda w: MAX_TURNOVER - 0.5 * np.sum(np.abs(w - prev_weights))
        })

    bounds = [(0, MAX_WEIGHT)] * n

    result = minimize(
        fun=lambda w: compute_tracking_error(w, portfolio_bonds, benchmark_stats),
        x0=w0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'maxiter': 500, 'ftol': 1e-9}
    )

    weights = result.x
    weights = np.maximum(weights, 0)
    weights /= weights.sum()  # normalise
    return weights, result

# Build initial portfolio
eligible  = select_eligible_bonds(bonds)
portfolio = stratified_sample(eligible, PORTFOLIO_SIZE)
weights, opt_result = optimise_portfolio(portfolio)

portfolio['weight'] = weights
portfolio['market_value'] = weights * PORTFOLIO_NAV
portfolio['notional']     = (portfolio.market_value / portfolio.price) * 100

print(f'✅ Portfolio optimised — {opt_result.message}')
print(f'   Bonds selected:    {len(portfolio)}')
print(f'   Optimiser status:  {"Converged" if opt_result.success else "Sub-optimal"}')
print(f'   Tracking Error:    {compute_tracking_error(weights, portfolio, {"duration": BENCHMARK_DURATION, "ytm": BENCHMARK_YTM, "dv01": BENCHMARK_DV01}):.2f} bps')

✅ Portfolio optimised — Optimization terminated successfully
   Bonds selected:    149
   Optimiser status:  Converged
   Tracking Error:    202.21 bps


In [6]:
#  Step 6: Portfolio Analytics

def portfolio_analytics(port_df, weights, nav=PORTFOLIO_NAV):
    """Compute full suite of fixed income portfolio analytics"""

    # Weighted averages
    port_duration = np.dot(weights, port_df.mod_duration)
    port_ytm      = np.dot(weights, port_df.ytm_pct)
    port_coupon   = np.dot(weights, port_df.coupon_pct)
    port_oas      = np.dot(weights, port_df.oas_bps)
    port_price    = np.dot(weights, port_df.price)

    # DV01: sum of position-level DV01s
    position_notionals = weights * nav / port_df.price * 100
    port_dv01_total = np.dot(position_notionals / 1000, port_df.dv01)

    # Tracking error
    bm_stats = {'duration': BENCHMARK_DURATION, 'ytm': BENCHMARK_YTM, 'dv01': BENCHMARK_DV01}
    te = compute_tracking_error(weights, port_df, bm_stats)

    # Duration mismatch
    duration_mismatch = port_duration - BENCHMARK_DURATION
    ytm_mismatch      = port_ytm - BENCHMARK_YTM

    # Sector weights
    sector_weights = {}
    for sector in BENCHMARK_SECTORS:
        mask = port_df.sector == sector
        sector_weights[sector] = float(weights[mask].sum())

    # Rating weights
    rating_weights = {}
    for rating in RATINGS:
        mask = port_df.rating == rating
        rating_weights[rating] = float(weights[mask].sum())

    return {
        'duration':          round(port_duration, 3),
        'duration_mismatch': round(duration_mismatch, 3),
        'ytm':               round(port_ytm, 3),
        'ytm_mismatch':      round(ytm_mismatch, 3),
        'coupon':            round(port_coupon, 3),
        'oas_bps':           round(port_oas, 1),
        'price':             round(port_price, 2),
        'dv01_total':        round(port_dv01_total, 0),
        'tracking_error_bps':round(te, 2),
        'n_bonds':           len(port_df),
        'sector_weights':    sector_weights,
        'rating_weights':    rating_weights,
        'max_weight':        round(float(weights.max()), 4),
        'herfindahl':        round(float(np.sum(weights**2)), 4),  # concentration
    }

analytics = portfolio_analytics(portfolio, weights)

print('📊 PORTFOLIO ANALYTICS')
print('═' * 45)
print(f'  Modified Duration:    {analytics["duration"]}y  (Benchmark: {BENCHMARK_DURATION}y | Δ: {analytics["duration_mismatch"]:+.3f}y)')
print(f'  Yield to Maturity:    {analytics["ytm"]}%   (Benchmark: {BENCHMARK_YTM}% | Δ: {analytics["ytm_mismatch"]:+.3f}%)')
print(f'  OAS Spread:           {analytics["oas_bps"]} bps')
print(f'  Average Coupon:       {analytics["coupon"]}%')
print(f'  Portfolio DV01:       ${analytics["dv01_total"]:,.0f}')
print(f'  Tracking Error:       {analytics["tracking_error_bps"]} bps p.a.')
print(f'  Number of Bonds:      {analytics["n_bonds"]} (of {BENCHMARK_N_BONDS} in benchmark)')
print(f'  Max Single Position:  {analytics["max_weight"]*100:.2f}%')
print(f'  Herfindahl Index:     {analytics["herfindahl"]:.4f} (lower = more diversified)')
print()
print('  Sector Weights vs Benchmark:')
for s, w in analytics['sector_weights'].items():
    bm = BENCHMARK_SECTORS[s]
    print(f'    {s:<15} Port: {w*100:5.1f}%  BM: {bm*100:5.1f}%  Δ: {(w-bm)*100:+.1f}%')

📊 PORTFOLIO ANALYTICS
═════════════════════════════════════════════
  Modified Duration:    8.416y  (Benchmark: 8.42y | Δ: -0.004y)
  Yield to Maturity:    9.354%   (Benchmark: 5.31% | Δ: +4.044%)
  OAS Spread:           100.8 bps
  Average Coupon:       9.5%
  Portfolio DV01:       $8,417
  Tracking Error:       202.21 bps p.a.
  Number of Bonds:      149 (of 2800 in benchmark)
  Max Single Position:  2.00%
  Herfindahl Index:     0.0196 (lower = more diversified)

  Sector Weights vs Benchmark:
    Financials      Port:  29.0%  BM:  28.0%  Δ: +1.0%
    Industrials     Port:  17.0%  BM:  22.0%  Δ: -5.0%
    Utilities       Port:  13.0%  BM:   8.0%  Δ: +5.0%
    Consumer        Port:  12.0%  BM:  14.0%  Δ: -2.0%
    Technology      Port:  17.0%  BM:  12.0%  Δ: +5.0%
    Healthcare      Port:   8.0%  BM:   9.0%  Δ: -1.0%
    Government      Port:   4.0%  BM:   7.0%  Δ: -3.0%


In [7]:
#  Step 7: Monthly Rebalancing Simulation (12 Months)
# Simulate index rebalancing events with auto-generated order lists
# Each month: benchmark weights shift slightly, bonds added/removed, optimiser re-runs

def simulate_index_drift(bonds_df, month):
    """Simulate benchmark index changes each month (additions, removals, weight shifts)"""
    drifted = bonds_df.copy()
    # Prices drift with market moves
    price_shock = np.random.normal(0, 0.8, len(drifted))  # ~0.8% monthly vol
    drifted['price'] = np.clip(drifted.price * (1 + price_shock/100), 70, 135)
    # Yields adjust
    rate_move = np.random.normal(0, 0.05)  # 5bps vol
    drifted['ytm_pct'] = np.clip(drifted.ytm_pct + rate_move, 1.0, 10.0)
    drifted['mod_duration'] = np.clip(drifted.mod_duration - 1/12, 0.5, 25)  # time decay
    return drifted

def generate_order_list(prev_weights, new_weights, portfolio_bonds, nav):
    """Generate buy/sell/hold order list from weight changes"""
    weight_diff = new_weights - prev_weights
    orders = []
    for i, (bond_id, diff) in enumerate(zip(portfolio_bonds.bond_id, weight_diff)):
        mv_change = diff * nav
        if abs(mv_change) > 50_000:  # Only trade if > $50k change
            action = 'BUY' if diff > 0 else 'SELL'
            orders.append({
                'bond_id':      bond_id,
                'sector':       portfolio_bonds.iloc[i].sector,
                'rating':       portfolio_bonds.iloc[i].rating,
                'action':       action,
                'weight_chg':   round(diff * 100, 4),
                'mv_change':    round(mv_change, 0),
                'prev_weight':  round(prev_weights[i] * 100, 4),
                'new_weight':   round(new_weights[i] * 100, 4),
            })
    return pd.DataFrame(orders).sort_values('mv_change', key=abs, ascending=False)

# Run 12-month simulation
monthly_results = []
current_weights = weights.copy()
current_portfolio = portfolio.copy()
prev_nav = PORTFOLIO_NAV

all_order_lists = []

for month in range(1, 13):
    # Drift the market
    drifted_bonds = simulate_index_drift(current_portfolio, month)

    # Re-optimise
    new_weights, _ = optimise_portfolio(drifted_bonds, prev_weights=current_weights)

    # Analytics
    stats = portfolio_analytics(drifted_bonds, new_weights)

    # Turnover
    turnover = 0.5 * np.sum(np.abs(new_weights - current_weights))

    # Simulated NAV (drift from yield carry + price changes)
    monthly_return = np.dot(new_weights, drifted_bonds.ytm_pct) / 12 / 100
    monthly_return += np.random.normal(0, 0.003)  # add noise
    new_nav = prev_nav * (1 + monthly_return)

    # Generate order list
    orders = generate_order_list(current_weights, new_weights, drifted_bonds, new_nav)
    orders['month'] = month
    all_order_lists.append(orders)

    monthly_results.append({
        'month':            month,
        'nav':              round(new_nav, 0),
        'monthly_return':   round(monthly_return * 100, 4),
        'duration':         stats['duration'],
        'duration_mismatch':stats['duration_mismatch'],
        'ytm':              stats['ytm'],
        'tracking_error':   stats['tracking_error_bps'],
        'turnover_pct':     round(turnover * 100, 2),
        'oas_bps':          stats['oas_bps'],
        'n_trades':         len(orders),
        'sector_weights':   stats['sector_weights'],
        'rating_weights':   stats['rating_weights'],
    })

    current_weights   = new_weights
    current_portfolio = drifted_bonds
    prev_nav          = new_nav

results_df = pd.DataFrame(monthly_results)
all_orders = pd.concat(all_order_lists, ignore_index=True)

print('✅ 12-month rebalancing simulation complete')
print(f'\n📋 REBALANCING SUMMARY')
print('═' * 50)
print(f'  Avg Monthly Tracking Error:  {results_df.tracking_error.mean():.2f} bps')
print(f'  Avg Monthly Turnover:        {results_df.turnover_pct.mean():.2f}%')
print(f'  Total Trades Generated:      {len(all_orders)}')
print(f'  Avg Trades/Month:            {len(all_orders)/12:.0f}')
print(f'  Final NAV:                   ${results_df.nav.iloc[-1]:,.0f}')
print(f'  Total Return:                {(results_df.nav.iloc[-1]/PORTFOLIO_NAV - 1)*100:.2f}%')
print()
print('Month-by-Month:')
print(results_df[['month','nav','monthly_return','duration','tracking_error','turnover_pct','n_trades']].to_string(index=False))

✅ 12-month rebalancing simulation complete

📋 REBALANCING SUMMARY
══════════════════════════════════════════════════
  Avg Monthly Tracking Error:  200.97 bps
  Avg Monthly Turnover:        0.80%
  Total Trades Generated:      44
  Avg Trades/Month:            4
  Final NAV:                   $109,373,420
  Total Return:                9.37%

Month-by-Month:
 month         nav  monthly_return  duration  tracking_error  turnover_pct  n_trades
     1 100567894.0          0.5679     8.415          203.30          1.53         6
     2 101422156.0          0.8494     8.414          202.48          2.04         6
     3 101719051.0          0.2927     8.413          202.74          0.88         3
     4 102742708.0          1.0064     8.413          199.47          1.01         4
     5 103489199.0          0.7266     8.413          199.73          0.90         4
     6 104585827.0          1.0597     8.413          203.09          0.39         3
     7 105116506.0          0.5074     8.413

In [8]:
#  Step 8: Risk Analytics — DV01, Sharpe, VaR

def compute_sharpe(monthly_returns, rf_monthly=0.0435/12):
    excess = np.array(monthly_returns) - rf_monthly
    return np.sqrt(12) * excess.mean() / excess.std() if excess.std() > 0 else 0

def compute_var(monthly_returns, confidence=0.95):
    """Historical VaR"""
    return np.percentile(monthly_returns, (1-confidence)*100)

def compute_max_drawdown(nav_series):
    nav = np.array(nav_series)
    peak = np.maximum.accumulate(nav)
    dd = (nav - peak) / peak
    return dd.min()

monthly_returns = results_df.monthly_return.values / 100
sharpe   = compute_sharpe(monthly_returns)
var_95   = compute_var(monthly_returns)
max_dd   = compute_max_drawdown(results_df.nav.values)
ann_ret  = (1 + np.mean(monthly_returns))**12 - 1
ann_vol  = np.std(monthly_returns) * np.sqrt(12)

# Benchmark tracking (simulate benchmark returns)
bm_monthly = np.random.normal(BENCHMARK_YTM/12/100, 0.003, 12)
tracking_diff = monthly_returns - bm_monthly
ann_tracking_error = np.std(tracking_diff) * np.sqrt(12) * 100  # in bps
information_ratio  = (np.mean(tracking_diff) / np.std(tracking_diff)) * np.sqrt(12) if np.std(tracking_diff) > 0 else 0

print('📊 RISK & PERFORMANCE ANALYTICS')
print('═' * 45)
print(f'  Annualised Return:       {ann_ret*100:.2f}%')
print(f'  Annualised Volatility:   {ann_vol*100:.2f}%')
print(f'  Sharpe Ratio:            {sharpe:.3f}')
print(f'  Information Ratio:       {information_ratio:.3f}')
print(f'  Annualised TE:           {ann_tracking_error:.1f} bps')
print(f'  VaR (95%, 1-month):      {var_95*100:.2f}%')
print(f'  Maximum Drawdown:        {max_dd*100:.2f}%')
print(f'  Portfolio DV01 (final):  ${analytics["dv01_total"]:,.0f}')
print(f'  Duration Mismatch (avg): {results_df.duration_mismatch.mean():+.3f}y')

📊 RISK & PERFORMANCE ANALYTICS
═════════════════════════════════════════════
  Annualised Return:       9.38%
  Annualised Volatility:   0.81%
  Sharpe Ratio:            5.762
  Information Ratio:       3.100
  Annualised TE:           1.4 bps
  VaR (95%, 1-month):      0.41%
  Maximum Drawdown:        0.00%
  Portfolio DV01 (final):  $8,417
  Duration Mismatch (avg): -0.007y


In [9]:
#  Step 9: Visualisations

# 1. NAV Performance
nav_with_bm = PORTFOLIO_NAV * np.cumprod(1 + bm_monthly)
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=list(range(1,13)), y=results_df.nav,
    name='Portfolio NAV', line=dict(color='#00d4aa', width=2.5)))
fig1.add_trace(go.Scatter(x=list(range(1,13)), y=nav_with_bm,
    name='Benchmark NAV', line=dict(color='#4a9eff', width=2, dash='dash')))
fig1.update_layout(title='Portfolio vs Benchmark NAV ($100M)', template='plotly_dark',
    xaxis_title='Month', yaxis_title='NAV ($)')
fig1.show()

# 2. Tracking Error over time
fig2 = go.Figure()
fig2.add_trace(go.Bar(x=list(range(1,13)), y=results_df.tracking_error,
    marker_color='#ff6b6b', name='Tracking Error (bps)'))
fig2.add_hline(y=results_df.tracking_error.mean(), line_dash='dash', line_color='white',
    annotation_text=f'Avg: {results_df.tracking_error.mean():.1f} bps')
fig2.update_layout(title='Monthly Tracking Error (bps)', template='plotly_dark')
fig2.show()

# 3. Duration Drift
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=list(range(1,13)), y=results_df.duration,
    name='Portfolio Duration', line=dict(color='#ffd700', width=2)))
fig3.add_hline(y=BENCHMARK_DURATION, line_dash='dash', line_color='#4a9eff',
    annotation_text=f'Benchmark: {BENCHMARK_DURATION}y')
fig3.add_hrect(y0=BENCHMARK_DURATION-0.5, y1=BENCHMARK_DURATION+0.5,
    fillcolor='rgba(74,158,255,0.1)', line_width=0, annotation_text='±0.5y band')
fig3.update_layout(title='Modified Duration vs Benchmark', template='plotly_dark')
fig3.show()

# 4. Sector allocation vs benchmark
final_sector_weights = results_df.iloc[-1]['sector_weights']
sectors = list(BENCHMARK_SECTORS.keys())
bm_w   = [BENCHMARK_SECTORS[s]*100 for s in sectors]
port_w = [final_sector_weights[s]*100 for s in sectors]

fig4 = go.Figure(data=[
    go.Bar(name='Portfolio', x=sectors, y=port_w, marker_color='#00d4aa'),
    go.Bar(name='Benchmark', x=sectors, y=bm_w,   marker_color='#4a9eff', opacity=0.7),
])
fig4.update_layout(barmode='group', title='Sector Allocation vs Benchmark (%)',
    template='plotly_dark')
fig4.show()

# 5. Monthly turnover
fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=list(range(1,13)), y=results_df.turnover_pct, mode='lines+markers',
    line=dict(color='#ff9f40', width=2), fill='tozeroy', fillcolor='rgba(255,159,64,0.15)'))
fig5.add_hline(y=MAX_TURNOVER*100, line_dash='dash', line_color='red',
    annotation_text='15% cap')
fig5.update_layout(title='Monthly Portfolio Turnover (%)', template='plotly_dark')
fig5.show()

print('✅ All visualisations rendered')

✅ All visualisations rendered


In [10]:
#  Step 10: Sample Order List — Month 1

month1_orders = all_orders[all_orders.month == 1].head(20)
print('📋 REBALANCING ORDER LIST — MONTH 1 (Top 20 by Size)')
print('═' * 75)
print(f'{"Bond ID":<12} {"Sector":<15} {"Rating":<8} {"Action":<6} {"Weight Δ":<12} {"MV Change ($)":<15} {"New Wt%"}')
print('─' * 75)
for _, row in month1_orders.iterrows():
    action_str = f'\033[92m{row.action}\033[0m' if row.action=='BUY' else f'\033[91m{row.action}\033[0m'
    print(f'{row.bond_id:<12} {row.sector:<15} {row.rating:<8} {row.action:<6} {row.weight_chg:>+8.4f}%   ${row.mv_change:>12,.0f}   {row.new_weight:.4f}%')

buys  = month1_orders[month1_orders.action=='BUY']
sells = month1_orders[month1_orders.action=='SELL']
print(f'\n  Total Buys:  {len(buys)} orders  |  ${buys.mv_change.sum():,.0f}')
print(f'  Total Sells: {len(sells)} orders  |  ${abs(sells.mv_change.sum()):,.0f}')

📋 REBALANCING ORDER LIST — MONTH 1 (Top 20 by Size)
═══════════════════════════════════════════════════════════════════════════
Bond ID      Sector          Rating   Action Weight Δ     MV Change ($)   New Wt%
───────────────────────────────────────────────────────────────────────────
BOND_0108    Utilities       A        SELL    -1.0000%   $  -1,005,679   1.0000%
BOND_0048    Utilities       A        BUY     +1.0000%   $   1,005,679   2.0000%
BOND_0460    Financials      AA       SELL    -0.3685%   $    -370,638   0.6315%
BOND_0205    Government      AAA      BUY     +0.3685%   $     370,638   0.3685%
BOND_0111    Technology      AA       SELL    -0.1589%   $    -159,834   1.0000%
BOND_0242    Technology      BBB      BUY     +0.1589%   $     159,834   2.0000%

  Total Buys:  3 orders  |  $1,536,151
  Total Sells: 3 orders  |  $1,536,151


In [11]:
#  Step 11: Export for Dashboard
import json

dashboard_data = {
    'portfolio_analytics': analytics,
    'monthly_results': results_df.to_dict(orient='records'),
    'performance': {
        'annual_return':    round(ann_ret*100, 2),
        'annual_vol':       round(ann_vol*100, 2),
        'sharpe_ratio':     round(sharpe, 3),
        'information_ratio':round(information_ratio, 3),
        'ann_tracking_error':round(ann_tracking_error, 1),
        'var_95':           round(var_95*100, 3),
        'max_drawdown':     round(max_dd*100, 2),
        'total_return':     round((results_df.nav.iloc[-1]/PORTFOLIO_NAV-1)*100, 2),
        'final_nav':        int(results_df.nav.iloc[-1]),
    },
    'benchmark': {
        'duration': BENCHMARK_DURATION,
        'ytm':      BENCHMARK_YTM,
        'dv01':     BENCHMARK_DV01,
        'n_bonds':  BENCHMARK_N_BONDS,
        'sector_weights': BENCHMARK_SECTORS,
        'rating_weights': BENCHMARK_RATINGS,
    },
    'nav_series':      results_df.nav.tolist(),
    'bm_nav_series':   nav_with_bm.tolist(),
    'tracking_error':  results_df.tracking_error.tolist(),
    'turnover':        results_df.turnover_pct.tolist(),
    'duration_series': results_df.duration.tolist(),
    'top_holdings':    portfolio.nlargest(10,'weight')[['bond_id','issuer','sector','rating','weight','mod_duration','ytm_pct','oas_bps']].round(4).to_dict(orient='records'),
    'order_list_m1':   all_orders[all_orders.month==1].head(15).to_dict(orient='records'),
}

with open('dashboard_data.json', 'w') as f:
    json.dump(dashboard_data, f, indent=2, default=str)

portfolio.to_csv('portfolio_holdings.csv', index=False)
results_df.to_csv('monthly_rebalancing.csv', index=False)
all_orders.to_csv('order_lists.csv', index=False)
bonds.to_csv('bond_universe.csv', index=False)

print('✅ All files exported:')
print('   dashboard_data.json     — for HTML dashboard')
print('   portfolio_holdings.csv  — current positions')
print('   monthly_rebalancing.csv — 12-month history')
print('   order_lists.csv         — all trade orders')
print('   bond_universe.csv       — full 500-bond universe')
print()
print('🎯 PROJECT COMPLETE')
print('─' * 50)
print(f'  Bonds in portfolio:        {PORTFOLIO_SIZE} of {N_BONDS} universe')
print(f'  Optimisation method:       SciPy SLSQP')
print(f'  Constraints enforced:      7')
print(f'  Rebalancing events:        12 (monthly)')
print(f'  Avg Tracking Error:        {results_df.tracking_error.mean():.1f} bps p.a.')
print(f'  Sharpe Ratio:              {sharpe:.3f}')
print(f'  Information Ratio:         {information_ratio:.3f}')
print(f'  Total Return (12 months):  {(results_df.nav.iloc[-1]/PORTFOLIO_NAV-1)*100:.2f}%')

✅ All files exported:
   dashboard_data.json     — for HTML dashboard
   portfolio_holdings.csv  — current positions
   monthly_rebalancing.csv — 12-month history
   order_lists.csv         — all trade orders
   bond_universe.csv       — full 500-bond universe

🎯 PROJECT COMPLETE
──────────────────────────────────────────────────
  Bonds in portfolio:        150 of 500 universe
  Optimisation method:       SciPy SLSQP
  Constraints enforced:      7
  Rebalancing events:        12 (monthly)
  Avg Tracking Error:        201.0 bps p.a.
  Sharpe Ratio:              5.762
  Information Ratio:         3.100
  Total Return (12 months):  9.37%
